# Bagging & Random Forest From Scratch
### Using only Python + Pandas (no sklearn, no AI libs)

**What we build:**
- A full Decision Tree (ID3) — self contained
- Bagging — train many trees on bootstrap samples, majority vote
- Random Forest — bagging + random feature subset at each split

**Key question we answer:** Why does bagging fail on tiny data, and what does a larger dataset change?

---
## Step 1 — Imports & the Decision Tree (Self-Contained)

In [ ]:
import pandas as pd
import math
import random
from collections import Counter

# ── Core math ────────────────────────────────────────────────────────────────

def entropy(series):
    probs = series.value_counts() / len(series)
    return -sum(p * math.log2(p) for p in probs if p > 0)

def information_gain(df, feature, target):
    n = len(df)
    wce = sum(len(g) / n * entropy(g[target]) for _, g in df.groupby(feature))
    return entropy(df[target]) - wce

# ── Node ─────────────────────────────────────────────────────────────────────

class Node:
    def __init__(self):
        self.feature    = None
        self.children   = {}
        self.prediction = None
        self.is_leaf    = False

# ── Build tree ───────────────────────────────────────────────────────────────
# `feature_subset` is used by Random Forest — only consider this many features
# at each node instead of all features. None = use all (standard bagging / single tree).

def build_tree(df, features, target, feature_subset=None):
    node = Node()

    if df[target].nunique() == 1:
        node.is_leaf    = True
        node.prediction = df[target].iloc[0]
        return node

    if not features:
        node.is_leaf    = True
        node.prediction = df[target].mode()[0]
        return node

    # ── Random Forest hook ───────────────────────────────────────────────────
    # Instead of evaluating ALL features, randomly pick a subset.
    # This is the ONLY line that separates Random Forest from plain Bagging.
    candidate_features = features
    if feature_subset is not None and feature_subset < len(features):
        candidate_features = random.sample(features, feature_subset)
    # ─────────────────────────────────────────────────────────────────────────

    best = max(candidate_features, key=lambda f: information_gain(df, f, target))
    node.feature  = best
    remaining     = [f for f in features if f != best]

    for value, subset in df.groupby(best):
        node.children[value] = build_tree(subset, remaining, target, feature_subset)

    return node

# ── Predict ──────────────────────────────────────────────────────────────────

def predict(node, row):
    if node.is_leaf:
        return node.prediction
    value = row[node.feature]
    if value not in node.children:
        return None          # unseen value — handled by ensemble majority vote
    return predict(node.children[value], row)

print('Decision Tree code ready.')

---
## Step 2 — Bootstrap Sampling

**Bootstrap** = sample N rows **with replacement** from a dataset of N rows.

On average, each bootstrap sample contains **~63%** unique rows.  
The remaining ~37% never appear — these are called **Out-of-Bag (OOB)** rows.

On a 14-row dataset, 63% = ~9 unique rows per tree.  
Trees trained on near-identical 9-row subsets will make near-identical splits — **almost no diversity**.

In [ ]:
def bootstrap_sample(df):
    """
    Sample N rows with replacement from df.
    Returns (bootstrap_df, oob_indices).
    OOB = rows NOT selected — used as a free validation set.
    """
    n = len(df)
    sampled_indices = [random.randint(0, n - 1) for _ in range(n)]
    oob_indices     = list(set(range(n)) - set(sampled_indices))
    return df.iloc[sampled_indices].reset_index(drop=True), oob_indices


# ── Demonstrate on Play Tennis (14 rows) ─────────────────────────────────────
random.seed(42)
small_df = pd.read_csv('weather_play.csv')

print(f'Original dataset size : {len(small_df)} rows')
print()
print('Bootstrap sample demo (3 runs):')
print('-' * 45)
for i in range(3):
    boot, oob = bootstrap_sample(small_df)
    unique    = boot.drop_duplicates().shape[0]
    print(f'  Run {i+1}: {unique} unique rows in bootstrap,  {len(oob)} OOB rows')

print()
print('Theoretical unique rows in bootstrap: ~63% of 14 =', round(0.632 * 14, 1))
print()
print('→ Trees trained on nearly identical data = very similar trees = bagging barely helps.')

---
## Step 3 — Bagging on Small Data (Play Tennis, 14 rows)

**Bagging** (Bootstrap AGGregatING):
1. Draw B bootstrap samples
2. Train one tree on each
3. Predict by **majority vote** across all trees

In [ ]:
def train_bagging(df, features, target, n_trees=10, feature_subset=None, seed=42):
    """
    Train a bagging ensemble.
    feature_subset=None  → plain Bagging (all features considered at each node)
    feature_subset=k     → Random Forest (only k random features at each node)
    Returns list of (tree, oob_indices) tuples.
    """
    random.seed(seed)
    ensemble = []
    for _ in range(n_trees):
        boot_df, oob_idx = bootstrap_sample(df)
        tree = build_tree(boot_df, features, target, feature_subset)
        ensemble.append((tree, oob_idx))
    return ensemble


def ensemble_predict(ensemble, row):
    """Majority vote across all trees for one row."""
    votes = [predict(tree, row) for tree, _ in ensemble]
    votes = [v for v in votes if v is not None]   # drop None (unseen values)
    if not votes:
        return None
    return Counter(votes).most_common(1)[0][0]


def ensemble_accuracy(ensemble, df, target):
    preds   = df.apply(lambda r: ensemble_predict(ensemble, r), axis=1)
    correct = (preds == df[target]).sum()
    return correct / len(df)


# ── Single tree baseline ──────────────────────────────────────────────────────
features_small = ['outlook', 'temperature', 'humidity', 'windy']
single_tree    = build_tree(small_df, features_small, 'play')
single_preds   = small_df.apply(lambda r: predict(single_tree, r), axis=1)
single_acc     = (single_preds == small_df['play']).mean()

# ── Bagging ensemble ──────────────────────────────────────────────────────────
bag_small = train_bagging(small_df, features_small, 'play', n_trees=20)
bag_acc   = ensemble_accuracy(bag_small, small_df, 'play')

print('Results on Play Tennis (14 rows):')
print('-' * 40)
print(f'  Single tree accuracy : {single_acc*100:.1f}%')
print(f'  Bagging (20 trees)   : {bag_acc*100:.1f}%')
print()
print('→ Little to no improvement because bootstrap samples are too similar.')
print('  Trees see almost the same data → they learn almost the same splits.')

---
## Step 4 — Create a Larger Synthetic Dataset (~200 rows)

We generate a synthetic dataset where the **true rules** are known:
- `income=high` and `credit=good`  → approved
- `income=low`                     → denied
- otherwise                        → depends on age and employment

We add **10% random noise** so no single tree can perfectly memorise it.

In [ ]:
random.seed(0)

incomes     = ['low', 'medium', 'high']
credits     = ['poor', 'fair', 'good']
ages        = ['young', 'middle', 'senior']
employments = ['employed', 'self_employed', 'unemployed']

rows = []
for _ in range(200):
    income     = random.choice(incomes)
    credit     = random.choice(credits)
    age        = random.choice(ages)
    employment = random.choice(employments)

    # True label rule
    if income == 'high' and credit == 'good':
        label = 'approved'
    elif income == 'low':
        label = 'denied'
    elif employment == 'unemployed':
        label = 'denied'
    elif age == 'young' and credit == 'poor':
        label = 'denied'
    else:
        label = 'approved'

    # 10% noise — randomly flip the label
    if random.random() < 0.10:
        label = 'denied' if label == 'approved' else 'approved'

    rows.append({'income': income, 'credit': credit, 'age': age,
                 'employment': employment, 'loan': label})

large_df = pd.DataFrame(rows)

# Train / test split — 80/20
large_df   = large_df.sample(frac=1, random_state=1).reset_index(drop=True)
split      = int(0.8 * len(large_df))
train_df   = large_df.iloc[:split].reset_index(drop=True)
test_df    = large_df.iloc[split:].reset_index(drop=True)

print(f'Large dataset  : {len(large_df)} rows')
print(f'Training set   : {len(train_df)} rows')
print(f'Test set       : {len(test_df)} rows')
print()
print('Label distribution (full):')
print(large_df['loan'].value_counts())

---
## Step 5 — Bagging on Large Data

With 200 rows, bootstrap samples are much more diverse.  
Each tree sees a meaningfully different subset → trees disagree more → majority vote is more powerful.

In [ ]:
features_large = ['income', 'credit', 'age', 'employment']
target_large   = 'loan'

# Single tree
single_large  = build_tree(train_df, features_large, target_large)
single_preds  = test_df.apply(lambda r: predict(single_large, r), axis=1)
single_acc    = (single_preds == test_df[target_large]).mean()

# Bagging — vary number of trees
print('Bagging accuracy vs number of trees (large dataset):')
print('-' * 45)
print(f'  Single tree (no bagging) : {single_acc*100:.1f}%')
print()

for n in [5, 10, 20, 50, 100]:
    bag   = train_bagging(train_df, features_large, target_large, n_trees=n)
    acc   = ensemble_accuracy(bag, test_df, target_large)
    print(f'  Bagging  {n:>3} trees        : {acc*100:.1f}%')

print()
print('→ Accuracy improves and stabilises as more trees are added.')
print('  Each tree sees ~126 unique rows (63% of 160) — diverse enough to matter.')

---
## Step 6 — Out-of-Bag (OOB) Error

A free bonus of bagging: every row is **absent** from ~37% of bootstrap samples.  
For each row, we can predict using **only the trees that never saw it** — a built-in validation set, no train/test split needed.

$$\text{OOB Error} = \frac{\text{rows where OOB prediction is wrong}}{\text{total rows}}$$

In [ ]:
def oob_error(ensemble, df, target):
    """
    For each row i, collect votes only from trees whose OOB indices include i.
    If a row was in every bootstrap sample (rare but possible), skip it.
    """
    oob_votes = {i: [] for i in range(len(df))}

    for tree, oob_indices in ensemble:
        for i in oob_indices:
            pred = predict(tree, df.iloc[i])
            if pred is not None:
                oob_votes[i].append(pred)

    correct = 0
    evaluated = 0
    for i, votes in oob_votes.items():
        if not votes:
            continue   # row appeared in every bootstrap — skip
        majority = Counter(votes).most_common(1)[0][0]
        if majority == df.iloc[i][target]:
            correct += 1
        evaluated += 1

    return 1 - (correct / evaluated), evaluated


# Compute OOB on the full large dataset (no train/test split needed)
bag_full = train_bagging(large_df, features_large, target_large, n_trees=100, seed=7)
oob_err, n_evaluated = oob_error(bag_full, large_df, target_large)

# Compare with test set accuracy
test_acc = ensemble_accuracy(bag_full, test_df, target_large)

print('OOB Error vs Test Error (large dataset, 100 trees):')
print('-' * 45)
print(f'  OOB error        : {oob_err*100:.1f}%   (evaluated on {n_evaluated} rows)')
print(f'  Test set error   : {(1-test_acc)*100:.1f}%   (evaluated on {len(test_df)} rows)')
print()
print('→ OOB error is a good approximation of test error — for free.')
print()

# Now show OOB on SMALL data to show it is noisy
bag_small_oob = train_bagging(small_df, features_small, 'play', n_trees=100, seed=7)
oob_err_small, n_eval_small = oob_error(bag_small_oob, small_df, 'play')
print('OOB Error on Play Tennis (14 rows, 100 trees):')
print('-' * 45)
print(f'  OOB error        : {oob_err_small*100:.1f}%   (evaluated on {n_eval_small} rows)')
print()
print('→ With 14 rows, few rows are truly OOB → noisy and unreliable estimate.')

---
## Step 7 — Random Forest

Random Forest = Bagging + **one extra trick**:

> At each node, instead of considering ALL features, consider only a **random subset of k features**.

This forces trees to be different from each other even when they see similar bootstrap samples.  
More diversity → better ensemble.

Typical rule of thumb for k:
- Classification: $k = \sqrt{\text{total features}}$
- Regression: $k = \text{total features} / 3$

**The only code difference from Bagging:**
```python
# Bagging
train_bagging(df, features, target, n_trees=50, feature_subset=None)

# Random Forest
train_bagging(df, features, target, n_trees=50, feature_subset=2)   # k=2
```

In [ ]:
import math as _math

n_features = len(features_large)                        # 4
k          = max(1, round(_math.sqrt(n_features)))      # sqrt(4) = 2

print(f'Total features : {n_features}')
print(f'Features per node (k = sqrt(n)) : {k}')
print()

# Compare Single Tree vs Bagging vs Random Forest
results = []

# Single tree
t = build_tree(train_df, features_large, target_large)
preds = test_df.apply(lambda r: predict(t, r), axis=1)
results.append({'Model': 'Single Tree', 'Test Accuracy': (preds == test_df[target_large]).mean()})

# Bagging
for n in [10, 50, 100]:
    bag = train_bagging(train_df, features_large, target_large, n_trees=n, feature_subset=None)
    acc = ensemble_accuracy(bag, test_df, target_large)
    results.append({'Model': f'Bagging ({n} trees)', 'Test Accuracy': acc})

# Random Forest
for n in [10, 50, 100]:
    rf  = train_bagging(train_df, features_large, target_large, n_trees=n, feature_subset=k)
    acc = ensemble_accuracy(rf, test_df, target_large)
    results.append({'Model': f'Random Forest ({n} trees, k={k})', 'Test Accuracy': acc})

results_df = pd.DataFrame(results)
results_df['Test Accuracy %'] = (results_df['Test Accuracy'] * 100).round(1)
results_df = results_df.drop(columns='Test Accuracy')
print(results_df.to_string(index=False))

---
## Step 8 — Feature Importance Across the Forest

For a single tree we walked every node and accumulated weighted IG.  
For a forest we do the same — but **average across all trees**.

This is more stable than a single tree's importance because it averages out the randomness introduced by bootstrap sampling and feature subsets.

In [ ]:
def node_importance(node, df_node, target, total_samples, importance):
    """Accumulate weighted IG for every internal node (same as single-tree step)."""
    if node.is_leaf or len(df_node) == 0:
        return
    f      = node.feature
    ig     = information_gain(df_node, f, target)
    weight = len(df_node) / total_samples
    importance[f] = importance.get(f, 0.0) + weight * ig

    for value, child in node.children.items():
        df_child = df_node[df_node[f] == value]
        node_importance(child, df_child, target, total_samples, importance)


def forest_feature_importance(ensemble, train_df, features, target):
    """
    Average feature importance across all trees in the ensemble.
    Each tree contributes its own weighted-IG importance.
    Final scores are normalised to sum to 1.
    """
    total_samples = len(train_df)
    accumulated   = {f: 0.0 for f in features}
    n_trees       = len(ensemble)

    for tree, _ in ensemble:
        tree_imp = {}
        node_importance(tree, train_df, target, total_samples, tree_imp)
        for f in features:
            accumulated[f] += tree_imp.get(f, 0.0)

    # Average across trees
    averaged = {f: accumulated[f] / n_trees for f in features}

    # Normalise to sum = 1
    total = sum(averaged.values())
    if total == 0:
        return {f: 0.0 for f in features}
    return {f: averaged[f] / total for f in features}


# Train a 100-tree RF and compute importance
rf_100 = train_bagging(train_df, features_large, target_large, n_trees=100, feature_subset=k)
imp    = forest_feature_importance(rf_100, train_df, features_large, target_large)

imp_df = pd.DataFrame([
    {'feature': f, 'importance': imp[f]} for f in features_large
]).sort_values('importance', ascending=False).reset_index(drop=True)

print('Random Forest — Feature Importance (averaged over 100 trees):')
print()
for _, row in imp_df.iterrows():
    bar = '█' * int(row['importance'] * 60)
    print(f"  {row['feature']:<15} {row['importance']:.4f}  {bar}")
print(f"\n  Sum = {imp_df['importance'].sum():.4f}")

---
## Step 9 — Side-by-Side: Small vs Large Data

Everything in one place to make the contrast clear.

In [ ]:
random.seed(42)

# ── Small dataset (Play Tennis, 14 rows) ──────────────────────────────────────
st_small  = build_tree(small_df, features_small, 'play')
st_acc    = (small_df.apply(lambda r: predict(st_small, r), axis=1) == small_df['play']).mean()

bag_s     = train_bagging(small_df, features_small, 'play', n_trees=100)
bag_s_acc = ensemble_accuracy(bag_s, small_df, 'play')

rf_s      = train_bagging(small_df, features_small, 'play', n_trees=100, feature_subset=2)
rf_s_acc  = ensemble_accuracy(rf_s, small_df, 'play')

# ── Large dataset (Loan, 160 train / 40 test) ─────────────────────────────────
st_large  = build_tree(train_df, features_large, target_large)
st_l_acc  = (test_df.apply(lambda r: predict(st_large, r), axis=1) == test_df[target_large]).mean()

bag_l     = train_bagging(train_df, features_large, target_large, n_trees=100)
bag_l_acc = ensemble_accuracy(bag_l, test_df, target_large)

rf_l      = train_bagging(train_df, features_large, target_large, n_trees=100, feature_subset=k)
rf_l_acc  = ensemble_accuracy(rf_l, test_df, target_large)

summary = pd.DataFrame([
    {'Dataset': 'Play Tennis (14 rows)',  'Model': 'Single Tree',        'Accuracy %': round(st_acc   * 100, 1)},
    {'Dataset': 'Play Tennis (14 rows)',  'Model': 'Bagging (100 trees)','Accuracy %': round(bag_s_acc* 100, 1)},
    {'Dataset': 'Play Tennis (14 rows)',  'Model': 'Random Forest (100)','Accuracy %': round(rf_s_acc * 100, 1)},
    {'Dataset': 'Loan (160/40 split)',    'Model': 'Single Tree',        'Accuracy %': round(st_l_acc * 100, 1)},
    {'Dataset': 'Loan (160/40 split)',    'Model': 'Bagging (100 trees)','Accuracy %': round(bag_l_acc* 100, 1)},
    {'Dataset': 'Loan (160/40 split)',    'Model': 'Random Forest (100)','Accuracy %': round(rf_l_acc * 100, 1)},
])

print(summary.to_string(index=False))
print()
print('Key takeaways:')
print('  1. Small data  → bagging and RF give little benefit over a single tree')
print('  2. Larger data → both ensembles clearly outperform a single tree')
print('  3. RF >= Bagging because feature randomness forces more diverse trees')

---
## Summary

| Concept | What it means |
|---|---|
| **Bootstrap sample** | N rows sampled with replacement — ~63% unique rows |
| **OOB rows** | The ~37% not selected — used as a free validation set |
| **Bagging** | Train B trees on B bootstrap samples, majority vote |
| **Diversity** | Trees must disagree for ensembles to work — needs enough data |
| **Random Forest** | Bagging + random k features at each node — forces diversity |
| **OOB Error** | Free accuracy estimate without a separate test set |
| **Forest importance** | Weighted IG per node, averaged across all trees |

### The one-line difference between Bagging and Random Forest
```python
candidate_features = random.sample(features, k)   # RF: add this line
# vs
candidate_features = features                      # Bagging: use all features
```

### What this notebook does NOT cover (next steps)
- **Continuous features** — splitting on thresholds, not categories
- **Regression trees** — using variance reduction instead of entropy
- **Max depth / min samples** — pruning individual trees in the forest
- **Gradient Boosting** — trees trained sequentially on errors, not in parallel